# ROGII - Wellbore Geology Prediction - Stage 4b submission notebook

Self-contained: no internet access needed (Code Competition requirement).

**Stage 4b: 1D CNN sequence model.** Convolves over a local window of consecutive
eval-zone rows (instead of Stage 4a's row-independent gradient boosting), using only
`GR` (the actual sequence signal), the three Stage 2/3 model-output signals
(`linear_prior`, `windowed_match`, `match_minus_prior`), and `eval_zone_frac` (position
within the eval zone, 0-1, the same range for every well).

**Three attempts were tried locally (30-well GroupKFold sample) before this one:**
1. All 11 Stage 4a features (incl. raw `MD/X/Y/Z`) -> RMSE 81.02. Train loss collapsed to
   near-zero within 1 epoch - the CNN was memorizing per-well identity from absolute
   location, not learning transferable GR-shape patterns.
2. Dropped `X/Y`, added dropout + weight decay -> RMSE 84.67 (regressed). `MD`,
   `dist_from_known_boundary`, `known_zone_rows` still leaked well identity almost as well.
3. **This config** - stripped every well-identifying scalar, kept only `GR` + the model-
   output signals + `eval_zone_frac` -> RMSE **70.18**. Real improvement over attempts 1-2,
   but still clearly worse than **Stage 4a's 52.90 local / 45.196 public LB**.

Full write-up in `../context/05-plan-of-attack.md` Stage 4b section. This notebook submits
attempt 3 anyway (per an explicit request for a real, honest data point at this stage) -
**it is expected to score worse than Stage 4a**. It is not the recommended model going
forward; Stage 4a remains the best submitted approach until Stage 5's ensemble.

In [ ]:
import glob
import os
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LinearRegression
from torch.utils.data import DataLoader, Dataset

RANDOM_STATE = 42
HALF_WINDOW = 5           # Stage 3 windowed GR match window
SEARCH_EXTRA_FT = 100.0
ACCEPT_SLACK = 1.5
CNN_WINDOW = 41            # Stage 4b sequence window (rows along MD)
CNN_FEATURE_COLS = ["GR", "linear_prior", "windowed_match", "match_minus_prior", "eval_zone_frac"]
EPOCHS = 1                 # bounded for runtime - epoch-1 vs epoch-2 was marginal locally

_KAGGLE_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
DATA_DIR = _KAGGLE_DIR if os.path.isdir(_KAGGLE_DIR) else os.path.join("..", "data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
SAMPLE_SUBMISSION = os.path.join(DATA_DIR, "sample_submission.csv")

print(f"DATA_DIR = {DATA_DIR}")
assert os.path.isdir(TRAIN_DIR), f"train dir not found: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR), f"test dir not found: {TEST_DIR}"

In [ ]:
def list_wells(split_dir):
    files = glob.glob(os.path.join(split_dir, "*__horizontal_well.csv"))
    return sorted(os.path.basename(f).split("__")[0] for f in files)


def linear_prior_predict(known, eval_rows):
    if len(known) < 2:
        fallback = known["TVT_input"].mean() if len(known) else np.nan
        return np.full(len(eval_rows), fallback)
    model = LinearRegression()
    model.fit(known[["MD", "Z"]], known["TVT_input"])
    return model.predict(eval_rows[["MD", "Z"]])


def windowed_shape_match(prior_preds, eval_gr, tw_tvt, tw_gr,
                          half_win=HALF_WINDOW, search_extra=SEARCH_EXTRA_FT,
                          accept_slack=ACCEPT_SLACK):
    n = len(prior_preds)
    if n == 0:
        return prior_preds
    gr_filled = pd.Series(eval_gr).interpolate(limit_direction="both").to_numpy()
    if np.isnan(gr_filled).all():
        return prior_preds
    refined = prior_preds.copy()
    match_err = np.full(n, np.nan)
    for i in range(n):
        lo_row, hi_row = max(0, i - half_win), min(n, i + half_win + 1)
        local_gr = gr_filled[lo_row:hi_row]
        L = len(local_gr)
        if np.isnan(local_gr).any() or L < 3:
            continue
        center_prior = prior_preds[i]
        lo_idx = np.searchsorted(tw_tvt, center_prior - search_extra)
        hi_idx = np.searchsorted(tw_tvt, center_prior + search_extra)
        if hi_idx - lo_idx < L:
            continue
        seg_gr = tw_gr[lo_idx:hi_idx]
        seg_tvt = tw_tvt[lo_idx:hi_idx]
        windows = np.lib.stride_tricks.sliding_window_view(seg_gr, L)
        sse = np.sum((windows - local_gr[None, :]) ** 2, axis=1)
        best = int(np.argmin(sse))
        center_offset = i - lo_row
        refined[i] = seg_tvt[best + center_offset]
        match_err[i] = sse[best] / L
    valid = ~np.isnan(match_err)
    if valid.sum() == 0:
        return prior_preds
    thresh = np.nanmedian(match_err[valid]) * accept_slack
    keep = valid & (match_err <= thresh)
    return np.where(keep, refined, prior_preds)


ALL_FEATURE_COLS = [
    "MD", "X", "Y", "Z", "GR", "linear_prior", "windowed_match",
    "match_minus_prior", "dist_from_known_boundary", "eval_zone_frac",
    "known_zone_rows",
]


def build_well_features(well, split_dir, has_target):
    hz = pd.read_csv(os.path.join(split_dir, f"{well}__horizontal_well.csv")).reset_index(drop=True)
    tw_path = os.path.join(split_dir, f"{well}__typewell.csv")
    tw = pd.read_csv(tw_path).dropna(subset=["TVT", "GR"]).sort_values("TVT")

    known = hz[hz["TVT_input"].notna()]
    eval_rows = hz[hz["TVT_input"].isna()]
    if len(eval_rows) == 0:
        return None

    linear_prior = linear_prior_predict(known, eval_rows)

    if len(tw) >= HALF_WINDOW * 2 + 1:
        tw_tvt = tw["TVT"].to_numpy()
        tw_gr = tw["GR"].to_numpy()
        eval_gr = eval_rows["GR"].to_numpy()
        windowed_match = windowed_shape_match(linear_prior, eval_gr, tw_tvt, tw_gr)
    else:
        windowed_match = linear_prior.copy()

    known_md_max = known["MD"].max() if len(known) else eval_rows["MD"].min()
    n_eval = len(eval_rows)

    df = pd.DataFrame({
        "well": well,
        "row_idx": eval_rows.index.to_numpy(),
        "MD": eval_rows["MD"].to_numpy(),
        "X": eval_rows["X"].to_numpy(),
        "Y": eval_rows["Y"].to_numpy(),
        "Z": eval_rows["Z"].to_numpy(),
        "GR": eval_rows["GR"].to_numpy(),
        "linear_prior": linear_prior,
        "windowed_match": windowed_match,
        "match_minus_prior": windowed_match - linear_prior,
        "dist_from_known_boundary": eval_rows["MD"].to_numpy() - known_md_max,
        "eval_zone_frac": (np.arange(n_eval) + 1) / n_eval,
        "known_zone_rows": len(known),
    })

    if has_target and "TVT" in hz.columns:
        df["target"] = eval_rows["TVT"].to_numpy()

    return df


def build_dataset(split_dir, wells, has_target):
    frames, failed = [], []
    for well in wells:
        try:
            f = build_well_features(well, split_dir, has_target)
            if f is not None:
                frames.append(f)
        except Exception as exc:  # noqa: BLE001
            print(f"  well {well} failed ({exc}); skipping")
            failed.append(well)
    if not frames:
        return pd.DataFrame(), failed
    return pd.concat(frames, ignore_index=True), failed

In [ ]:
class WellWindowDataset(Dataset):
    def __init__(self, wells_features, wells_targets, index_list, window=CNN_WINDOW):
        self.wells_features = wells_features
        self.wells_targets = wells_targets
        self.index_list = index_list
        self.window = window
        self.half = window // 2

    def __len__(self):
        return len(self.index_list)

    def __getitem__(self, i):
        well, row_idx = self.index_list[i]
        feats = self.wells_features[well]
        n = feats.shape[0]
        lo, hi = row_idx - self.half, row_idx + self.half + 1
        idx = np.clip(np.arange(lo, hi), 0, n - 1)
        window = feats[idx].T
        y = self.wells_targets[well][row_idx] if self.wells_targets is not None else 0.0
        return torch.from_numpy(window.copy()), torch.tensor(y, dtype=torch.float32)


class CNN1D(nn.Module):
    def __init__(self, in_channels=len(CNN_FEATURE_COLS), dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, padding=2), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(32, 32, kernel_size=5, padding=2), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(32, 32, kernel_size=5, padding=2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h = self.net(x).squeeze(-1)
        return self.fc(h).squeeze(-1)


def build_per_well_arrays(dataset_df, target_col=None):
    wells_features, wells_targets = {}, ({} if target_col else None)
    for well, group in dataset_df.groupby("well", sort=False):
        wells_features[well] = group[CNN_FEATURE_COLS].to_numpy(dtype=np.float32)
        if target_col:
            wells_targets[well] = group[target_col].to_numpy(dtype=np.float32)
    return wells_features, wells_targets

In [ ]:
t0 = time.time()
train_wells = list_wells(TRAIN_DIR)
print(f"Building TRAIN features for {len(train_wells)} wells...")
train_data, train_failed = build_dataset(TRAIN_DIR, train_wells, has_target=True)
train_data = train_data.dropna(subset=["target"])
print(f"Train dataset: {train_data.shape}, {len(train_failed)} wells failed, "
      f"built in {time.time()-t0:.1f}s")

# Impute GR NaN (~39% missing) and standardize every CNN feature - raw scale
# and NaNs both broke early attempts. Target also standardized (raw ~11-12k
# magnitude made the bias converge far too slowly from random init).
train_data["GR"] = train_data["GR"].fillna(train_data["GR"].median())
feature_stats = {}
for col in CNN_FEATURE_COLS:
    mean, std = train_data[col].mean(), train_data[col].std()
    std = std if std > 1e-6 else 1.0
    train_data[col] = (train_data[col] - mean) / std
    feature_stats[col] = (mean, std)

target_mean, target_std = train_data["target"].mean(), train_data["target"].std()
train_data["target_scaled"] = (train_data["target"] - target_mean) / target_std
GLOBAL_FALLBACK = float(target_mean)
print(f"target_mean={target_mean:.2f} target_std={target_std:.2f}")

In [ ]:
# Train the FINAL model on ALL training data (bounded to 1 epoch for runtime -
# the 30-well local test showed epoch-1 vs epoch-2 gains were marginal).
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

wells_features, wells_targets = build_per_well_arrays(train_data, target_col="target_scaled")
full_index = [(w, i) for w, arr in wells_targets.items() for i in range(len(arr))]

train_ds = WellWindowDataset(wells_features, wells_targets, full_index)
train_dl = DataLoader(train_ds, batch_size=512, shuffle=True, num_workers=0)

model = CNN1D().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(EPOCHS):
    model.train()
    total_loss, n_seen = 0.0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item() * len(yb)
        n_seen += len(yb)
    print(f"epoch {epoch}: train MSE {total_loss/n_seen:.4f}")

print(f"Model trained on {len(train_data)} rows. Elapsed: {time.time()-t0:.1f}s")

In [ ]:
test_wells = list_wells(TEST_DIR)
print(f"Building TEST features for {len(test_wells)} wells...")
test_data, test_failed = build_dataset(TEST_DIR, test_wells, has_target=False)
print(f"Test dataset: {test_data.shape}, {len(test_failed)} wells failed")

rows = []
if len(test_data) > 0:
    test_data = test_data.reset_index(drop=True)
    test_data["GR"] = test_data["GR"].fillna(test_data["GR"].median())
    for col in CNN_FEATURE_COLS:
        mean, std = feature_stats[col]
        test_data[col] = (test_data[col] - mean) / std
    test_data[CNN_FEATURE_COLS] = test_data[CNN_FEATURE_COLS].fillna(0.0)

    test_features, _ = build_per_well_arrays(test_data, target_col=None)
    row_idx_in_well = test_data.groupby("well").cumcount().to_numpy()
    test_index = list(zip(test_data["well"], row_idx_in_well))
    test_ds = WellWindowDataset(test_features, None, test_index)
    test_dl = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=0)

    model.eval()
    preds_scaled = []
    try:
        with torch.no_grad():
            for xb, _ in test_dl:
                xb = xb.to(device)
                preds_scaled.append(model(xb).cpu().numpy())
        preds_scaled = np.concatenate(preds_scaled)
        preds = preds_scaled * target_std + target_mean
    except Exception as exc:  # noqa: BLE001
        print(f"batch predict failed ({exc}); falling back to GLOBAL_FALLBACK for all test rows")
        preds = np.full(len(test_data), GLOBAL_FALLBACK)

    for well, row_idx, pred in zip(test_data["well"], test_data["row_idx"], preds):
        rows.append({"id": f"{well}_{row_idx}", "tvt": pred})

submission = pd.DataFrame(rows, columns=["id", "tvt"])
print(f"built {len(submission)} predictions across "
      f"{len(test_wells) - len(test_failed)} wells ({len(test_failed)} wells failed)")
submission.head()

In [ ]:
sample = pd.read_csv(SAMPLE_SUBMISSION)
submission = submission.drop_duplicates(subset="id", keep="first")

merged = sample[["id"]].merge(submission, on="id", how="left")
n_missing = merged["tvt"].isna().sum()
if n_missing > 0:
    print(f"WARNING: {n_missing} required ids had no prediction - filling with "
          f"GLOBAL_FALLBACK ({GLOBAL_FALLBACK:.2f})")
    merged["tvt"] = merged["tvt"].fillna(GLOBAL_FALLBACK)

extra_ids = set(submission["id"]) - set(sample["id"])
if extra_ids:
    print(f"NOTE: {len(extra_ids)} predicted ids aren't in sample_submission - dropped")

assert len(merged) == len(sample), f"row count still off: {len(merged)} vs {len(sample)}"
assert merged["tvt"].notna().all(), "still have NaNs after fallback fill - should be impossible"

submission = merged
submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv:", submission.shape)
print(f"Total runtime: {time.time()-t0:.1f}s")